# ManūAI · 03 — Evaluate and explore errors

- Report for a finished run (segment- and recording-level, AGENTS.md §8)
- Normalised confusion matrix and per-class F1
- **Listen to the mistakes**: the most confident wrong predictions on the test set
- Try the model on any audio file

In [ ]:
%load_ext autoreload
%autoreload 2
import json, os
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, Audio, display

from manuai import features as F
from manuai.config import REPO_ROOT, data_dir
from manuai.labels import class_names
os.chdir(REPO_ROOT)
CLASSES = class_names()
print(f"repo: {REPO_ROOT}\nclasses ({len(CLASSES)}): {', '.join(CLASSES)}")

In [ ]:
runs = sorted((p.parent for p in (REPO_ROOT / "models/finetuned").glob("*/training_metrics.json")),
              key=lambda p: p.stat().st_mtime)
print("runs:", [r.name for r in runs])
RUN = runs[-1] if runs else None          # or e.g. REPO_ROOT / "models/finetuned/ast-v2-1"
summary = json.loads((RUN / "training_metrics.json").read_text())
FEATURES = summary["args"]["features"]
print("run:", RUN.name, "· features:", FEATURES)

In [ ]:
from manuai import report
display(Markdown(report.render(summary)))

## Full evaluation
The overnight job already evaluates `test`. Flip the flag to (re)run a split here (~1 h for the full test set on the M1).

In [ ]:
from manuai import evaluate
EVALUATE = False
SPLITS = ["test"]                       # add "val" if wanted
if EVALUATE:
    evaluate.main(["--run", str(RUN), "--features", FEATURES, "--splits", *SPLITS])
    summary = json.loads((RUN / "training_metrics.json").read_text())

## Confusion matrix and per-class F1

In [ ]:
SPLIT = "test"
rep = summary["eval"].get(SPLIT)
assert rep, f"no {SPLIT} evaluation in this run yet"
classes = summary["classes"]
cm = np.asarray(rep["confusion_matrix"], dtype=float)
cmn = cm / np.maximum(cm.sum(1, keepdims=True), 1)
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(classes)), classes, rotation=75, fontsize=8); ax.set_yticks(range(len(classes)), classes, fontsize=8)
for i in range(len(classes)):
    for j in range(len(classes)):
        if cmn[i, j] >= 0.05:
            ax.text(j, i, f"{cmn[i, j]:.2f}", ha="center", va="center", fontsize=6, color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(f"{RUN.name} · {SPLIT} · row-normalised")
plt.colorbar(im, fraction=0.04); plt.tight_layout(); plt.show()

f1 = pd.Series({c: rep["per_class"][c]["f1-score"] for c in classes}).sort_values()
ax = f1.plot.barh(figsize=(8, 7)); ax.axvline(rep["segment"]["macro_f1"], ls="--", c="k", label="macro-F1")
ax.set_xlim(0, 1); ax.legend(); ax.set_title("per-class F1 (segment level)"); plt.tight_layout(); plt.show()

## Listen to the mistakes
Most confident wrong predictions — often label noise, background species or tūī mimicry.

In [ ]:
from manuai.data import read_rows
probs_path = RUN / f"probs_{SPLIT}.npy"
assert probs_path.exists(), "run the evaluation first"
probs = np.load(probs_path).astype(np.float32)
rows = read_rows(data_dir() / FEATURES, SPLIT)
if len(rows) != len(probs):   # smoke runs evaluate a subset — rebuild it the same way
    from manuai.data import spread_subset
    rows = spread_subset(rows, summary["args"]["eval_per_class"], summary["args"]["seed"])
rows = rows.assign(pred=[classes[i] for i in probs.argmax(1)], conf=probs.max(1))
wrong = rows[rows.pred != rows.label].sort_values("conf", ascending=False)
print(f"{len(wrong):,} wrong of {len(rows):,} segments")
wrong[["label", "pred", "conf", "source", "recording_id", "path", "seg_no"]].head(15)

In [ ]:
feat = np.load(data_dir() / FEATURES / f"{SPLIT}.npy", mmap_mode="r")
for _, r in wrong.head(3).iterrows():
    plt.figure(figsize=(8, 2.2)); plt.imshow(np.asarray(feat[r.seg], np.float32).T, origin="lower", aspect="auto", cmap="magma")
    plt.title(f"true {r.label} → predicted {r.pred} ({r.conf:.0%}) · {r.source}"); plt.axis("off"); plt.show()
    display(Audio(str(REPO_ROOT / r.path)))

## Try any audio file

In [ ]:
from manuai.inference import load_run, pick_device, predict_file
AUDIO = None   # e.g. "~/Downloads/backyard_tui.m4a" (anything libsndfile can read: wav/flac/ogg/mp3)
if AUDIO:
    device = pick_device()
    model, spec, collate = load_run(RUN, device)
    res = predict_file(Path(AUDIO).expanduser(), model, spec, collate, device, top_k=5)
    display(Audio(str(Path(AUDIO).expanduser())))
    display(pd.DataFrame(res["top"]))
    print("per 5 s segment:", res["per_segment"])